# Baseline Models for Five-Year Overall Survival

This notebook prepares the SEER breast cancer dataset and trains two baseline classifiers: logistic regression and a decision tree. The prediction target is death before five years, using clinical and pathological features recorded at diagnosis or initial staging.

The models are starting points for the project. Predictive performance, explainability, fairness, and other FUTURE-AI criteria still need to be assessed. The held-out test set is reserved for the final evaluation of the selected model.


## 1. Imports and Reproducible Settings

Import the libraries used for data handling, preprocessing, splitting, and modeling. Set a fixed random seed so that the split and model initialization can be reproduced, and define the five-year prediction horizon as 60 months.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
HORIZON = 60  # months: 5 years
np.random.seed(SEED)
pd.set_option("display.max_columns", None)


## 2. Load the Dataset and Define the Outcome

The notebook first looks for the CSV in the local project folder and the current working directory. If it is not found and the notebook is running in Google Colab, it mounts Google Drive and checks the configured Drive path. Update that path if the file is stored elsewhere.

After removing the empty technical column and duplicate rows, create the target using the same rules as the original project notebook:

- **Class 1:** death before 60 months.
- **Class 0:** observed survival of at least 60 months, regardless of what happens later.
- **Excluded:** recorded alive with less than 60 months of follow-up; the outcome at five years is unknown.

Excluding these early censored patients makes the binary label observable, but may affect how representative the analyzed cohort is. The printed class proportions describe the remaining patients.


In [ ]:
CSV_NAME = "Copia di SEER Breast Cancer Dataset .csv"
PROJECT_DIR = Path(r"C:\Progetti\neuroengineering_course\progetto")
LOCAL_CANDIDATES = [PROJECT_DIR / CSV_NAME, Path.cwd() / CSV_NAME]
DATASET_PATH = next((path for path in LOCAL_CANDIDATES if path.exists()), None)

if DATASET_PATH is None:
    try:
        from google.colab import drive
    except ImportError as exc:
        searched = ", ".join(str(path) for path in LOCAL_CANDIDATES)
        raise FileNotFoundError(
            f"CSV not found. Local paths checked: {searched}. "
            "Place the file in the project folder or working directory."
        ) from exc

    drive.mount("/content/drive")
    DATASET_PATH = Path("/content/drive/MyDrive/progettoneuro") / CSV_NAME

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"CSV not found at {DATASET_PATH}. Check the Drive path."
    )

print(f"Dataset loaded from: {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH)
df = df.drop(columns=["Unnamed: 3"], errors="ignore")
df.columns = df.columns.str.strip()
df = df.drop_duplicates().reset_index(drop=True)
assert df.isna().sum().sum() == 0, "The dataset contains missing values that need to be handled."

survived_5y = df[df["Survival Months"] >= HORIZON].copy()
survived_5y["label"] = 0
died_before_5y = df[
    (df["Status"] == "Dead") & (df["Survival Months"] < HORIZON)
].copy()
died_before_5y["label"] = 1
censored_before_5y = df[
    (df["Status"] == "Alive") & (df["Survival Months"] < HORIZON)
]
task_df = pd.concat([survived_5y, died_before_5y], ignore_index=True)

print(f"Evaluable patients: {len(task_df)}")
print(f"Censored before five years and excluded: {len(censored_before_5y)}")
print("Class distribution (proportions):")
print(task_df["label"].value_counts(normalize=True).sort_index().round(3))


## 3. Separate Predictors and Create a Stratified Split

Create the feature matrix and target separately. Remove survival time, final status, and the target from the predictors because they directly reveal the outcome. Reserve 25% of the evaluable patients as a test set, and stratify the split to keep similar class proportions in training and test data. The fixed seed makes the split reproducible.


In [ ]:
LEAKAGE_COLS = ["Survival Months", "Status", "label"]
X = task_df.drop(columns=LEAKAGE_COLS)
y = task_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

print("Train:", X_train.shape, " Held-out test:", X_test.shape)
print(
    "Class 1 prevalence — train:", round(y_train.mean(), 3),
    " test:", round(y_test.mean(), 3)
)


## 4. Define Preprocessing for Each Feature Type

Standardize the numerical features and apply one-hot encoding to categorical features. Standardization is useful for scale-sensitive models such as logistic regression; it does not remove outliers. Unknown categories are ignored during transformation so that a new category does not cause an error.

T and N stages have a clinical order. One-hot encoding is used as a baseline because it does not assume equal numerical distances between stages; an ordinal encoding can be considered later. Each pipeline receives its own preprocessor, which is fitted with its model on training data.


In [ ]:
NUM_COLS = [
    "Age", "Tumor Size", "Regional Node Examined", "Reginol Node Positive"
]
CAT_COLS = [
    "Race", "Marital Status", "T Stage", "N Stage", "6th Stage",
    "Grade", "A Stage", "Estrogen Status", "Progesterone Status"
]
assert set(NUM_COLS) | set(CAT_COLS) == set(X.columns)

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ])


## 5. Build and Fit the Baseline Pipelines

A pipeline combines preprocessing and a classifier. Logistic regression provides a linear baseline; the decision tree can represent nonlinear relationships and interactions, but an unrestricted tree may overfit. Both models use balanced class weights, giving more weight to the less frequent outcome during training without resampling. This choice still needs to be assessed.

Fit both pipelines using only the training data. A successful fit confirms that the models can be trained; it does not establish how accurately or fairly they perform on new patients.


In [ ]:
logistic_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=SEED
    )),
])

decision_tree_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", DecisionTreeClassifier(
        class_weight="balanced", random_state=SEED
    )),
])

baseline_models = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": decision_tree_pipeline,
}
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    print(f"{name}: fitted on the training set ({len(X_train)} patients)")


## 6. Next Steps

The two fitted pipelines are available in baseline_models. Their performance has not yet been evaluated in this notebook. After model development and selection, evaluate the selected model on the reserved test set and examine explainability, fairness, and other relevant FUTURE-AI criteria.
